In [ ]:
!pip install "psycopg[binary]==3.3.6" "python-dotenv==1.2.4"


In [ ]:
import os
import psycopg
from psycopg import sql
from dotenv import load_dotenv

load_dotenv("../../.env")
PGHOST = os.getenv("PGHOST", "localhost")
PGPORT = os.getenv("PGPORT", "5432")
PGUSER = os.getenv("PGUSER", "postgres")
PGPASSWORD = os.getenv("PGPASSWORD") or None
PGDATABASE = os.getenv("PGDATABASE", "university")
PGSSLMODE = os.getenv("PGSSLMODE", "prefer")
conn_args = dict(host=PGHOST, port=PGPORT, user=PGUSER,
                 password=PGPASSWORD, sslmode=PGSSLMODE)

with psycopg.connect(autocommit=True, **{**conn_args, "dbname": "postgres"}) as admin:
    if not admin.execute("SELECT 1 FROM pg_database WHERE datname = %s", (PGDATABASE,)).fetchone():
        admin.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(PGDATABASE)))
conn = psycopg.connect(autocommit=True, dbname=PGDATABASE, **conn_args)
conn.execute("CREATE EXTENSION IF NOT EXISTS pgstattuple")
print("connected")


In [ ]:
def show(query, params=None):
    with conn.cursor() as cur:
        cur.execute(query, params)
        if cur.description is None:
            print(cur.statusmessage); return
        cols = [d.name for d in cur.description]
        rows = cur.fetchall()
    cells = [["NULL" if v is None else str(v) for v in row] for row in rows]
    widths = [max([len(c)] + [len(row[i]) for row in cells]) for i,c in enumerate(cols)]
    print(" | ".join(c.center(w) for c,w in zip(cols,widths)).rstrip())
    print("-+-".join("-"*w for w in widths))
    for row in cells:
        print(" | ".join(v.ljust(w) for v,w in zip(row,widths)).rstrip())
    print(f"({len(rows)} row{'' if len(rows)==1 else 's'})")


In [ ]:
conn.execute("DROP TABLE IF EXISTS biglog CASCADE")
conn.execute("CREATE TABLE biglog (id bigserial primary key, payload text, created_at timestamptz default now())")
conn.execute("INSERT INTO biglog (payload) SELECT repeat('x', 200) || g FROM generate_series(1, 20000) g")
conn.execute("CREATE INDEX idx_biglog_created ON biglog(created_at)")
conn.execute("ANALYZE biglog")

show("""
    SELECT relname, pg_size_pretty(pg_table_size(oid)) AS table_size,
           pg_size_pretty(pg_indexes_size(oid)) AS indexes_size,
           pg_size_pretty(pg_total_relation_size(oid)) AS total_size
    FROM pg_class WHERE relname IN ('biglog','idx_biglog_created')
    ORDER BY relname
""")

show("SELECT pg_size_pretty(pg_database_size(current_database())) AS db_size")


- **Input:** a freshly created table with an index.
- **What happens:** insert 20,000 rows with a 200-character payload, add an index on `created_at`, analyze, and measure sizes.
- **Output:** table, indexes, and total sizes in human-readable form.
- **Why:** you cannot cut costs until you measure what takes space.


In [ ]:
show("""
    SELECT schemaname, relname, indexrelname, idx_scan,
           pg_size_pretty(pg_relation_size(indexrelid)) AS index_size
    FROM pg_stat_user_indexes
    WHERE relname = 'biglog'
    ORDER BY indexrelname
""")


- **Input:** index usage statistics from `pg_stat_user_indexes`.
- **What happens:** look at `idx_scan` (number of times the index was used). A brand-new index with `idx_scan = 0` may be unused or just not used yet.
- **Output:** each index on `biglog` with its scan count and size.
- **Why:** drop indexes with zero scans in real workloads only after confirming they're not used in planned queries or constraints.

```mermaid
flowchart LR
    S["Check pg_stat_user_indexes"] --> Z{"idx_scan == 0?"}
    Z -->|yes| C{"Is it a constraint or<br/>about to be used?"}
    C -->|no| D["Consider dropping<br/>(saves space + writes)"]
    C -->|yes| K["Keep for now"]
    Z -->|no| K
```


In [ ]:
show("SELECT approx_free_percent, approx_tuple_count, dead_tuple_count FROM pgstattuple_approx('biglog')")


- **Input:** the `biglog` table.
- **What happens:** `pgstattuple_approx` gives a fast estimate of free space and tuple counts (approximate to avoid a full table scan).
- **Output:** percentage of free space and tuple counts.
- **Why:** before vacuuming you want a baseline; after churn you'll see more free space.


In [ ]:
# Create some dead tuples by deleting a fraction of the rows.
conn.execute("DELETE FROM biglog WHERE id % 10 = 0")
conn.execute("ANALYZE biglog")
show("SELECT approx_free_percent FROM pgstattuple_approx('biglog')")
conn.execute("VACUUM biglog")
show("SELECT approx_free_percent FROM pgstattuple_approx('biglog')")


- **Input:** deleted 10% of the rows.
- **What happens:** deleted rows become dead tuples; `VACUUM` marks them as removable and updates visibility so space can be reused.
- **Output:** `approx_free_percent` changes after `VACUUM` (in this small demo it may increase or behave differently depending on state).
- **Why:** `VACUUM` is normal maintenance. `VACUUM FULL` rewrites the table and takes an exclusive lock — use only when you must shrink the file size dramatically. Autovacuum handles most of this automatically.

```mermaid
flowchart LR
    U["UPDATE/DELETE"] --> D["Dead tuples appear"]
    D --> A{"Autovacuum runs<br/>or you run VACUUM"}
    A --> R["Space marked reusable<br/>visibility updated"]
    R --> I["Indexes also cleaned<br/>over time"]
```


In [ ]:
conn.execute("DROP TABLE IF EXISTS orders_bad, orders_good")
conn.execute("CREATE TABLE orders_bad (id bigserial primary key, status char(10), notes text, amount numeric)")
conn.execute("CREATE TABLE orders_good (id bigserial primary key, amount numeric(10,2), status text, notes text)")

show("""
    SELECT relname, pg_size_pretty(pg_table_size(oid)) AS table_size
    FROM pg_class WHERE relname IN ('orders_bad','orders_good')
""")


- **Input:** two near-identical tables with different types/ordering.
- **What happens:** create them empty and compare `pg_table_size`.
- **Output:** sizes for both tables (small here; the lesson shows the principle).
- **Why:** prefer `text` over `char(n)` unless a fixed length is required; use appropriately sized numerics/integers; put fixed/commonly accessed columns thoughtfully. Smaller rows mean more fit in memory and cheaper I/O.


In [ ]:
conn.execute("DROP TABLE IF EXISTS logs_2025_09, logs_2025_10")
conn.execute("CREATE TABLE logs_2025_09 (id bigserial, log_time timestamptz, payload text)")
conn.execute("CREATE TABLE logs_2025_10 (id bigserial, log_time timestamptz, payload text)")
conn.execute("INSERT INTO logs_2025_09 SELECT g, '2025-09-15', repeat('y',50) FROM generate_series(1,5000) g")
conn.execute("INSERT INTO logs_2025_10 SELECT g, '2025-10-01', repeat('y',50) FROM generate_series(1,5000) g")

show("""
    SELECT relname, pg_size_pretty(pg_total_relation_size(oid)) AS total_size
    FROM pg_class WHERE relname LIKE 'logs_%' ORDER BY relname
""")

# Archive September, then drop it (cheap vs deleting rows)
conn.execute("CREATE TABLE log_archive_2025_09 AS SELECT * FROM logs_2025_09")
conn.execute("DROP TABLE logs_2025_09")

show("SELECT relname FROM pg_class WHERE relname IN ('logs_2025_10','log_archive_2025_09') ORDER BY relname")


- **Input:** two month-sized log tables.
- **What happens:** copy old data to an archive table, then `DROP TABLE` the old partition/table.
- **Output:** sizes before, then the old table gone and archive present.
- **Why:** `DROP TABLE` removes the files quickly; `DELETE FROM` would touch every row, bloat, and take much longer. For partitions, `ALTER TABLE ... DETACH CONVERT TO TABLE` / dropping partitions is the standard cost-saving pattern.

```mermaid
flowchart LR
    M["Month partition<br/>becomes old"] --> A["Archive if needed<br/>(CREATE TABLE AS SELECT)"]
    A --> D["DROP TABLE<br/>the old partition"]
    D --> F["Space freed quickly"]
```


In [ ]:
show("SELECT current_setting('shared_buffers') AS shared_buffers, current_setting('work_mem') AS work_mem, current_setting('effective_cache_size') AS effective_cache_size")
show("SELECT numbackends, xact_commit, xact_rollback FROM pg_stat_database WHERE datname = current_database()")
show("SELECT count(*) AS active_backends FROM pg_stat_activity WHERE pid <> pg_backend_pid()")


- **Input:** current server settings and activity.
- **What happens:** inspect key memory settings and connection counts.
- **Output:** values for this server (tuned by the environment).
- **Why:** *Guidelines:* start from measured usage. `shared_buffers` caches data pages; `effective_cache_size` tells the planner how much OS cache to expect; `work_mem` is per sort/hash operation (multiply by operations/connections carefully). Too many connections = high memory use — use a pooler.

**Pooler note (PgBouncer):** put PgBouncer between apps and PostgreSQL. Modes: `session` (connection held for whole session), `transaction` (released after each transaction — most common for web apps), `statement` (released after each statement — restrictive). Fewer backend connections mean less memory and overhead.


In [ ]:
conn.execute("DROP TABLE IF EXISTS biglog, orders_bad, orders_good, logs_2025_10, log_archive_2025_09 CASCADE")
conn.close()
print("done")
